In [1]:
!pip uninstall -y azure-ai-ml marshmallow
!pip install "azure-ai-ml==1.32.0" "marshmallow<4" azure-identity

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.2/13.2 MB 57.9 MB/s  0:00:00m0:00:01
  Attempting uninstall: psutil
    Found existing installation: psutil 5.2.2━━━━━━━━━━━━━━━━━━━━━  1/29 [psutil]
    Uninstalling psutil-5.2.2:━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  1/29 [psutil]
      Successfully uninstalled psutil-5.2.2━━━━━━━━━━━━━━━━━━━  1/29 [psutil]
  Attempting uninstall: opentelemetry-api━━━━━━━━━━━━━━━━━━━━━━━━━  5/29 [strictyaml]]
    Found existing installation: opentelemetry-api 1.33.0━━━━━  5/29 [strictyaml]
    Uninstalling opentelemetry-api-1.33.0:━━━━━━━━━━━━━━━━━━━━  5/29 [strictyaml]
      Successfully uninstalled opentelemetry-api-1.33.0━━━━━━━━━━━  6/29 [opentelemetry-api]
  Attempting uninstall: opentelemetry-semantic-conventions━━━━━━━━  6/29 [opentelemetry-api]
    Found existing installation: opentelemetry-semantic-conventions 0.54b0/29 [opentelemetry-api]
    Uninstalling opentelemetry-semantic-conventions-0.54b0:━━━  6/29 [opentelemetry-api]
      Successfully unin

In [7]:
!pip show azure.identity
!pip show azure.ai.ml


Name: azure-identity
Version: 1.17.0
Summary: Microsoft Azure Identity Library for Python
Home-page: https://github.com/Azure/azure-sdk-for-python/tree/main/sdk/identity/azure-identity
Author: Microsoft Corporation
Author-email: azpysdkhelp@microsoft.com
License: MIT License
Location: /anaconda/envs/azureml_py38/lib/python3.10/site-packages
Requires: azure-core, cryptography, msal, msal-extensions, typing-extensions
Required-by: adlfs, azure-monitor-opentelemetry-exporter, azureml-dataprep, azureml-mlflow


In [2]:
from azure.identity import DefaultAzureCredential, InteractiveBrowserCredential
from azure.ai.ml import MLClient


try: 
    credential = DefaultAzureCredential()
    credential.get_token("https://management.azure.com/.default")
except:
    credential = InteractiveBrowserCredential()

In [3]:
ml_client= MLClient.from_config(credential=credential)

Found the config file in: /config.json
Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


In [15]:
stores = ml_client.datastores.list()
for ds_name in stores:
    print(ds_name.name)

blob_training_data
workspaceblobstore
workspaceworkingdirectory
workspacefilestore
workspaceartifactstore


In [14]:
from azure.ai.ml.entities import AzureBlobDatastore
from azure.ai.ml.entities import AccountKeyConfiguration

store= AzureBlobDatastore(
    name= "blob_training_data",
    description="data",
    account_name= "mspai1929623230",
    container_name="msp-data",
    credentials= AccountKeyConfiguration(account_key="deKI/MRuALY2FnnX+9SB8wDRM3f8/zJgNj7Uro+RXFFCuCXTd4YCO1BPCkLlv78Ll5hc8VjxJniW+AStnQBgvA==")
)
ml_client.create_or_update(store)

AzureBlobDatastore({'type': <DatastoreType.AZURE_BLOB: 'AzureBlob'>, 'name': 'blob_training_data', 'description': 'data', 'tags': {}, 'properties': {}, 'print_as_yaml': False, 'id': '/subscriptions/e56d44da-2d28-421c-b85d-423c8f6b5510/resourceGroups/rg-openclaw/providers/Microsoft.MachineLearningServices/workspaces/mspai/datastores/blob_training_data', 'Resource__source_path': '', 'base_path': '/mnt/batch/tasks/shared/LS_root/mounts/clusters/mspcompute/code/Users/ooyeneyin', 'creation_context': None, 'serialize': <msrest.serialization.Serializer object at 0x71773d3c9c60>, 'credentials': {'type': 'account_key'}, 'container_name': 'msp-data', 'account_name': 'mspai1929623230', 'endpoint': 'core.windows.net', 'protocol': 'https'})

In [18]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

data_path="azureml://datastores/blob_training_data/paths/data-asset-path"
data= Data(
    path= data_path,
    type= AssetTypes.URI_FOLDER,
    description="data asset",
    name="my_data",
)
ml_client.data.create_or_update(data)

Data({'path': 'azureml://subscriptions/e56d44da-2d28-421c-b85d-423c8f6b5510/resourcegroups/rg-openclaw/workspaces/mspai/datastores/blob_training_data/paths/data-asset-path/', 'skip_validation': False, 'mltable_schema_url': None, 'referenced_uris': None, 'type': 'uri_folder', 'is_anonymous': False, 'auto_increment_version': False, 'auto_delete_setting': None, 'name': 'my_data', 'description': 'data asset', 'tags': {}, 'properties': {}, 'print_as_yaml': False, 'id': '/subscriptions/e56d44da-2d28-421c-b85d-423c8f6b5510/resourceGroups/rg-openclaw/providers/Microsoft.MachineLearningServices/workspaces/mspai/data/my_data/versions/1', 'Resource__source_path': '', 'base_path': '/mnt/batch/tasks/shared/LS_root/mounts/clusters/mspcompute/code/Users/ooyeneyin', 'creation_context': <azure.ai.ml.entities._system_data.SystemData object at 0x7176ff4a8d30>, 'serialize': <msrest.serialization.Serializer object at 0x7176ff4aa170>, 'version': '1', 'latest_version': None, 'datastore': None})

In [33]:
%%writefile src/clean.py

import pandas as pd
import numpy as np
import argparse
import os
import math

parser = argparse.ArgumentParser()
parser.add_argument("--data", type=str, required=True)
parser.add_argument("--output", type=str, required=True)
args = parser.parse_args()

data_path = args.data
print(f"Data path: {data_path}")

s_file_path = os.path.join(data_path, "pump_specs_static_3rows.csv")
spec = pd.read_csv(s_file_path)

t_file_path = os.path.join(data_path, "pump_maintenance_timeseries_1000_rows.csv")
table = pd.read_csv(t_file_path)

output_dict = {
    "ID": [],
    "date": [],
    "pressure": [],
    "flow_rate": [],
    "suction_pressure": [],
    "discharge_pressure": [],
    "rpm": [],
    "vibration_rms": [],
    "vibration_peak": [],
    "structural_vibration": [],
    "motor_current": [],
    "power_draw_kw": [],
    "bearing_temperature": [],
    "motor_temperature": [],
    "pump_casing_temperature": [],
    "oil_temperature": [],
    "fluid_temperature": [],
    "ambient_temperature": [],
    "oil_pressure": [],
    "oil_level": [],
    "seal_pressure": [],
    "leakage_rate": [],
    "runtime_hours_since_last_maintenance": [],
    "days_since_last_maintenance": [],
    "start_stop_count": [],
    "acoustic_signal_level": [],
    "time_until_next_maintenance": []
}

for _, row in table.iterrows():
    match = spec[spec["ID"] == row["ID"]]
    if match.empty:
        continue

    data = match.iloc[0]

    pressure = row["pressure"] / (data["rho"] * 9.8 * data["Hr"])
    flow = row["flow_rate"] / data["Q"]

    output_dict["ID"].append(row["ID"])
    output_dict["date"].append(row["date"])
    output_dict["pressure"].append(pressure)
    output_dict["flow_rate"].append(flow)
    output_dict["suction_pressure"].append(row["suction_pressure"] / data["Ps"])
    output_dict["discharge_pressure"].append(row["discharge_pressure"] / data["Pd"])
    output_dict["rpm"].append(row["rpm"] / data["N"])
    output_dict["vibration_rms"].append(row["vibration_rms"] / data["Vrms"])
    output_dict["vibration_peak"].append(row["vibration_peak"] / data["Vpeak"])
    output_dict["structural_vibration"].append(row["structural_vibration"] / data["Vstruct"])
    output_dict["motor_current"].append(
        row["motor_current"] * math.sqrt(3) * data["V_r"] * data["PF"] / (data["P_motor_r"] * 1000)
    )
    output_dict["power_draw_kw"].append(
        row["power_draw_kw"] * data["eta_pump"] * data["eta_motor"] / (pressure * flow)
    )
    output_dict["bearing_temperature"].append(
        (row["bearing_temperature"] - row["ambient_temperature"]) / (data["T_bearing"] - row["ambient_temperature"])
    )
    output_dict["motor_temperature"].append(
        (row["motor_temperature"] - row["ambient_temperature"]) / (data["T_motor"] - row["ambient_temperature"])
    )
    output_dict["pump_casing_temperature"].append(
        (row["pump_casing_temperature"] - row["ambient_temperature"]) / (data["T_casing"] - row["ambient_temperature"])
    )
    output_dict["oil_temperature"].append(
        (row["oil_temperature"] - row["ambient_temperature"]) / (data["T_oil"] - row["ambient_temperature"])
    )
    output_dict["fluid_temperature"].append(row["fluid_temperature"] / data["T_fluid"])
    output_dict["ambient_temperature"].append(row["ambient_temperature"])
    output_dict["oil_pressure"].append(row["oil_pressure"] / data["P_oil"])
    output_dict["oil_level"].append(row["oil_level"] / data["Level_oil"])
    output_dict["seal_pressure"].append(row["seal_pressure"] / data["P_seal"])
    output_dict["leakage_rate"].append(row["leakage_rate"] / data["Leakage"])
    output_dict["runtime_hours_since_last_maintenance"].append(
        row["runtime_hours_since_last_maintenance"] / data["Runtime_since_maintenance"]
    )
    output_dict["days_since_last_maintenance"].append(
        row["days_since_last_maintenance"] / data["Days_since_maintenance"]
    )
    output_dict["start_stop_count"].append(row["start_stop_count"] / data["StartStop_allowed"])
    output_dict["acoustic_signal_level"].append(row["acoustic_signal_level"] / data["Acoustic_actual"])
    output_dict["time_until_next_maintenance"].append(row["time_until_next_maintenance"])

cleaned_data = pd.DataFrame(output_dict)

os.makedirs(args.output, exist_ok=True)
cleaned_data.to_csv(os.path.join(args.output, "processed.csv"), index=False)

Overwriting src/clean.py


In [37]:
from azure.ai.ml import command, Input, Output

job = command(
    code="./src",
    command="""
python clean.py \
  --data ${{inputs.data}} \
  --output ${{outputs.output_data}}
""",
    inputs={
        "data": Input(
            type="uri_folder",
            path="azureml://datastores/blob_training_data/paths/",
            mode="ro_mount"
        )
    },
    outputs={
        "output_data": Output(
            type="uri_folder",
            # optionally:
            path="azureml://datastores/blob_training_data/paths/",
            mode="rw_mount"
        )
    },
    environment="AzureML-sklearn-1.5:1",   # example; use one that actually exists in your workspace
    compute="mspcompute",
    display_name="mspAudit-cleaning",
    experiment_name="cleaning"
)

train_job = ml_client.jobs.create_or_update(job)
print(train_job.studio_url)

pathOnCompute is not a known attribute of class <class 'azure.ai.ml._restclient.v2023_04_01_preview.models._models_py3.UriFolderJobOutput'> and will be ignored


https://ml.azure.com/runs/clever_jackal_zj4sjxhlj8?wsid=/subscriptions/e56d44da-2d28-421c-b85d-423c8f6b5510/resourcegroups/rg-openclaw/workspaces/mspai&tid=7085725b-f01d-4d02-ad1c-b1a8a6f44106


In [ ]:
%%writefile environment.yaml
name: sklearn-tf
channels:
  - conda-forge

dependencies:
  - python=3.10
  - pip
  - pip:
      - numpy
      - pandas
      - matplotlib
      - scikit-learn==1.5.0
      - tensorflow==2.16.1
      - mlflow
      - azureml-mlflow==1.62.0.post2
      - inference-schema[numpy-support]==1.3.0
      - xlrd==2.0.1
      - psutil>=5.8,<5.9
      - tqdm>=4.59,<4.60
      - ipykernel~=6.0


In [ ]:
from azure.ai.ml.entities import Environment

ENV= "sklearn-tf"
env = Environment(
    name=ENV,
    conda_file="environment.yaml",
    image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu22.04:latest"
)

ml_client.environments.create_or_update(env)

Environment({'arm_type': 'environment_version', 'latest_version': None, 'image': 'mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu22.04:latest', 'intellectual_property': None, 'is_anonymous': False, 'auto_increment_version': False, 'auto_delete_setting': None, 'name': 'sklearn-tf', 'description': None, 'tags': {}, 'properties': {'azureml.labels': 'latest'}, 'print_as_yaml': False, 'id': '/subscriptions/e56d44da-2d28-421c-b85d-423c8f6b5510/resourceGroups/rg-openclaw/providers/Microsoft.MachineLearningServices/workspaces/mspai/environments/sklearn-tf/versions/3', 'Resource__source_path': '', 'base_path': '/mnt/batch/tasks/shared/LS_root/mounts/clusters/ooyeneyin1/code/Users/ooyeneyin', 'creation_context': <azure.ai.ml.entities._system_data.SystemData object at 0x7f2c54517160>, 'serialize': <msrest.serialization.Serializer object at 0x7f2c54516a40>, 'version': '3', 'conda_file': {'channels': ['conda-forge'], 'dependencies': ['python=3.10', 'pip', {'pip': ['numpy', 'pandas', 'matplotlib', 'sc

In [16]:
%%writefile src/training.py
#libraries

import os
import argparse
import numpy as np
import pandas as pd
import mlflow
import mlflow.tensorflow

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout


def create_sequences(data: np.ndarray, lookback: int):
    X, y = [], []
    for i in range(lookback, len(data)):
        X.append(data[i - lookback:i, 0])
        y.append(data[i, 0])
    return np.array(X), np.array(y)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--data", type=str, required=True, help="Directory containing processed.csv")
    parser.add_argument("--output", type=str, required=True, help="Output directory for saved model")
    args = parser.parse_args()

    data_path = args.data
    output_path = args.output
    csv_path = os.path.join(data_path, "processed.csv")

    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"Could not find input file: {csv_path}")

    print(f"Data path: {data_path}")
    print(f"Reading: {csv_path}")

    df = pd.read_csv(csv_path)

    required_columns = {"date", "time_until_next_maintenance"}
    missing = required_columns - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")

    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df = df.dropna(subset=["date", "time_until_next_maintenance"])

    if "ID" in df.columns:
        df = df.sort_values(["ID", "date"]).reset_index(drop=True)
    else:
        df = df.sort_values("date").reset_index(drop=True)

    data = df[["time_until_next_maintenance"]].astype(float)

    scaler = MinMaxScaler(feature_range=(0, 1))
    scaled_data = scaler.fit_transform(data)

    lookback = 90
    if len(scaled_data) <= lookback:
        raise ValueError(
            f"Not enough rows to build sequences. Need more than {lookback}, got {len(scaled_data)}."
        )

    X, y = create_sequences(scaled_data, lookback)

    # Reshape for LSTM: (samples, timesteps, features)
    X = X.reshape((X.shape[0], X.shape[1], 1))

    split_index = int(len(X) * 0.8)
    if split_index == 0 or split_index == len(X):
        raise ValueError("Train/test split failed. Check dataset size.")

    X_train, X_test = X[:split_index], X[split_index:]
    y_train, y_test = y[:split_index], y[split_index:]

    model = Sequential([
        LSTM(64, return_sequences=True, input_shape=(X_train.shape[1], 1)),
        Dropout(0.2),
        LSTM(32),
        Dropout(0.2),
        Dense(1),
    ])

    model.compile(optimizer="adam", loss="mse")

    os.makedirs(output_path, exist_ok=True)

    mlflow.tensorflow.autolog()

    with mlflow.start_run():
        history = model.fit(
            X_train,
            y_train,
            epochs=30,
            batch_size=32,
            validation_data=(X_test, y_test),
            verbose=1,
        )

        y_pred = model.predict(X_test)

        y_test_actual = scaler.inverse_transform(y_test.reshape(-1, 1))
        y_pred_actual = scaler.inverse_transform(y_pred)

        rmse = np.sqrt(mean_squared_error(y_test_actual, y_pred_actual))
        mae = mean_absolute_error(y_test_actual, y_pred_actual)
        r2 = r2_score(y_test_actual, y_pred_actual)

        print("R²:", r2)
        print("RMSE:", rmse)
        print("MAE:", mae)

        mlflow.log_metric("rmse", rmse)
        mlflow.log_metric("mae", mae)
        mlflow.log_metric("r2", r2)
        mlflow.log_param("lookback", lookback)
        local_model_path = os.path.join(output_path, "trained_model")
        mlflow.tensorflow.save_model(model=model, path=local_model_path)
        # Log model to MLflow tracking
        mlflow.log_artifacts(local_model_path, artifact_path="model")

        print(f"Model saved locally to: {local_model_path}")


if __name__ == "__main__":
    main()

Overwriting src/training.py


In [17]:
from azure.ai.ml import command, Input,Output

job= command(
    code="./src",
     command="""
python training.py \
  --data ${{inputs.data}} \
  --output ${{outputs.output_data}}
""",
    inputs={
        "data": Input(
            type="uri_folder",
            path="azureml://datastores/blob_training_data/paths/",
            mode="ro_mount"
        )
    },
        outputs={
        "output_data": Output(
            type="uri_folder",
            # optionally:
            path="azureml://datastores/blob_training_data/paths/",
            mode="rw_mount"
        )
        },
   environment="sklearn-tf@latest",   # example; use one that actually exists in your workspace
    compute="msp-audit",
    display_name="work",
    experiment_name="training"
)
train_job = ml_client.jobs.create_or_update(job)
print(train_job.studio_url)

Uploading src (0.01 MBs): 100%|██████████| 8617/8617 [00:00<00:00, 111683.07it/s]


pathOnCompute is not a known attribute of class <class 'azure.ai.ml._restclient.v2023_04_01_preview.models._models_py3.UriFolderJobOutput'> and will be ignored


https://ml.azure.com/runs/purple_cow_gkvpqgczdd?wsid=/subscriptions/e56d44da-2d28-421c-b85d-423c8f6b5510/resourcegroups/rg-openclaw/workspaces/mspai&tid=7085725b-f01d-4d02-ad1c-b1a8a6f44106


In [11]:
import uuid

# Creating a unique name for the endpoint
online_endpoint_name = "credit-endpoint-" + str(uuid.uuid4())[:8]

In [12]:
from azure.ai.ml.entities import (
    ManagedOnlineEndpoint,
    ManagedOnlineDeployment,
    Model,
    Environment,
)

# create an online endpoint
endpoint = ManagedOnlineEndpoint(
    name=online_endpoint_name,
    description="this is an online endpoint",
    auth_mode="key",
)

endpoint = ml_client.online_endpoints.begin_create_or_update(endpoint).result()

print(f"Endpoint {endpoint.name} provisioning state: {endpoint.provisioning_state}")

/anaconda/envs/azureml_py310_sdkv2/lib/python3.10/site-packages/mlflow/__init__.py:41: UserWarning: Versions of mlflow (3.8.1) and child packages mlflow-skinny (3.5.0) are different. This may lead to unexpected behavior. Please install the same version of all MLflow packages.
  mlflow.mismatch._check_version_mismatch()


Endpoint credit-endpoint-e78657d8 provisioning state: Succeeded


In [18]:
endpoint = ml_client.online_endpoints.get(name=online_endpoint_name)

print(
    f'Endpoint "{endpoint.name}" with provisioning state "{endpoint.provisioning_state}" is retrieved'
)

Endpoint "credit-endpoint-e78657d8" with provisioning state "Succeeded" is retrieved


In [21]:
latest_model_version = max(
    [int(m.version) for m in ml_client.models.list(name="msp-audit")]
)

In [ ]:
model = ml_client.models.get(name="msp-audit", version=latest_model_version)


# create an online deployment.
blue_deployment = ManagedOnlineDeployment(
    name="blue",
    endpoint_name=online_endpoint_name,
    model=model,
    environment=ENV,
    instance_type="Standard_DS2_v2",
    instance_count=1,
)

blue_deployment = ml_client.begin_create_or_update(blue_deployment).result()

Instance type Standard_DS2_v2 may be too small for compute resources. Minimum recommended compute SKU is Standard_DS3_v2 for general purpose endpoints. Learn more about SKUs here: https://learn.microsoft.com/azure/machine-learning/referencemanaged-online-endpoints-vm-sku-list
Check: endpoint credit-endpoint-e78657d8 exists


.................................................

In [ ]:
ml_client.online_endpoints.begin_delete(name=online_endpoint_name)